#### [ CNN CUSTOM DATASET + MODEL ]

- 사용자 정의 이미지 데이터셋 생성 ==> ImageFolder 사용
- 사용자 정의 cnn기반 모델 설계
- 데이터 : 강아지, 고양이 사진



[1] 모듈 로딩 및 데이터 준비 <hr>

In [2]:
## 모듈 로딩
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.utils
import torch.utils.data

from torchvision.datasets import ImageFolder            ## 이미지용 데이터셋 생성 모듈
from torch.utils.data import DataLoader                 ## 데이터로더
from torchvision.transforms import transforms           ## 이미지 전처리 및 증강 모듈


import matplotlib.pyplot as plt


In [3]:
## 데이터 준비
IMG_ROOT = '../_image/cat_dog/'
# IMG_TRAIN_ROOT = '../_image/cat_dog_train/'
# IMG_ROOT = '../_image/train1/'
# IMG_TRAIN_ROOT = '../_image/test1/'

[2] 데이터 로딩 및 데이터셋 준비 <hr>

In [4]:
# print(f"classes  => {imgDS.classes}")
# print(f"class_to_idx  => {imgDS.class_to_idx}")
# print(f"targets  => {imgDS.targets}")
# print(f"imgs  => {imgDS.imgs}")
# print(imgDS[0])

In [5]:
from PIL import Image

In [8]:
## 2-0 이미지 전처리 및 변형
## resize - > 이미지 크기 통일      ==> transforms.Resize((shape))
## tensor --> 텐서로 타입 변형.     ==> transforms.ToTensor() : 텐서화 + 정규화(0~1)
##                                  ==>                         채널(C, H, W)로 변경.
preprocessing = transforms.Compose(
    [
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((50,50)),
        transforms.ToTensor()
    ]
    )

# testDS = ImageFolder(IMG_ROOT, transform=preprocessing)
# trainDS = ImageFolder(IMG_TRAIN_ROOT, transform=preprocessing)
imgDS = ImageFolder(IMG_ROOT, transform=preprocessing)


In [11]:
## 방법 2) 층화 샘플링(stratified sampling)

## -------(1) 비율 기준 설정
## 모듈 로딩
from sklearn.model_selection import train_test_split  

## 전체 데이터 수 체크
print('[전체 데이터 수 체크 ]', len(imgDS.imgs), len(imgDS.targets))

## 비율 기준 설정
targets = imgDS.targets

## -------(2) 훈련용:검증용:테스트용 데이터 인덱스 추출
## 1단계: 학습용과 검증용 데이터셋 분리
train_indices, valid_indices = train_test_split(range(len(imgDS.targets)),
                                                test_size=0.2, random_state=42, 
                                                stratify=targets)

## 2단계: 검증용 데이터셋을 테스트용 데이터셋 분리
targets = [ targets[index] for index in valid_indices ]
valid_indices, test_indices= train_test_split( valid_indices,
                                               test_size=0.5, random_state=42, 
                                               stratify=targets)

## -------(3) 훈련용:검증용:테스트용 데이터셋 생성
trainDS = Subset(imgDS, train_indices)
validDS = Subset(imgDS, valid_indices)
testDS = Subset(imgDS, test_indices)

## -------(4) 분리 데이터셋의 클래스 비율 체크
for kind, indices in zip(['Train','Valid','Test'], [trainDS.indices, validDS.indices, testDS.indices]):
    targets = [ imgDS.targets[ idx ] for idx in indices ]
    print(f'\n[{kind} 전체 데이터셋 개수 : {len(targets)}개')
    print(f'        - cat      개수 : {targets.count(0):02}개 {(targets.count(0)/len(targets))*100:.2f}')
    print(f'        - dog      개수 : {targets.count(1):02}개 {(targets.count(1)/len(targets))*100:.2f}')

[전체 데이터 수 체크 ] 138 138


NameError: name 'Subset' is not defined

In [8]:
# ### CNN으로 캣/독 이진분류.

# class CD_CNN(nn.Module):
#     def __init__(self):
#         super().__init__()
#         ## 특징맵 추출 부분
#         self.con_layer1 = nn.Conv2d(3, 16, 3, padding=1)  # (1, 3, 50, 50) -> (1, 16, 50, 50)
#         self.pool_layer1 = nn.MaxPool2d(2, 2)             # (1, 16, 50, 50) -> (1, 16, 50, 50)
#         self.con_layer2 = nn.Conv2d(16, 32, 3, padding=1) # (1, 16, 50, 50) -> (1, 32, 50, 50)
#         self.pool_layer2 = nn.MaxPool2d(2, 2)             # (1, 32, 50, 50) -> (1, 32, 25, 25)
        
#         self.flat_layer = nn.Flatten()                    # (1, 32, 25, 25) -> (1, 32*25*25)
        
#         ## 전결합 학습 부분
#         self.fc_layer1 = nn.Linear(32*12*12, 512)
#         self.h_layer1 = nn.Linear(512, 128)
#         self.drop = nn.Dropout(0.25)  # Dropout2d 대신 Dropout 사용
#         self.h_layer2 = nn.Linear(128, 64)
#         self.h_layer3 = nn.Linear(64, 32)
#         self.out_layer = nn.Linear(32, 1)  # 이진 분류 (출력 노드 1개)
        
#     def forward(self, x):
#         x = self.pool_layer1(nn.ReLU()(self.con_layer1(x)))
#         x = self.pool_layer2(nn.ReLU()(self.con_layer2(x)))
#         x = self.flat_layer(x)
#         x = nn.ReLU()(self.fc_layer1(x))
#         x = nn.ReLU()(self.h_layer1(x))
#         x = self.drop(x)
#         x = nn.ReLU()(self.h_layer2(x))
#         x = nn.ReLU()(self.h_layer3(x))
#         x = self.out_layer(x)  
#         return x
        



In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CD_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        ## 특징맵 추출 부분
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)  # (B, 3, 50, 50) -> (B, 16, 50, 50)
        self.pool1 = nn.MaxPool2d(2, 2)                          # (B, 16, 50, 50) -> (B, 16, 25, 25)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1) # (B, 16, 25, 25) -> (B, 32, 25, 25)
        self.pool2 = nn.MaxPool2d(2, 2)                          # (B, 32, 25, 25) -> (B, 32, 12, 12)
        
        self.flatten = nn.Flatten()  # (B, 32, 12, 12) -> (B, 32 * 12 * 12)

        ## 전결합 학습 부분
        self.fc1 = nn.Linear(32 * 12 * 12, 512)  # 4608 -> 512
        self.dropout = nn.Dropout(0.25)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 64)
        self.dropout = nn.Dropout(0.25)
        self.fc4 = nn.Linear(64, 32)
        self.out = nn.Linear(32, 1)  # 이진 분류

        

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))

        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        x = F.relu(self.fc4(x))
        x = self.out(x)  

        return x  # BCEWithLogitsLoss() 사용 시, Sigmoid 적용 안 함.


In [10]:
test = CD_CNN()

In [11]:
import sys

sys.path.append('../_utils/')
from DL_Module import TT_classifier

In [12]:
testCF = TT_classifier(test, trainDS, testDS, 
                       batch_size=10,
                       LOSS_FN=nn.BCEWithLogitsLoss())

EPOCHS : 100
ITERATION : 3
LOSS_FN : BCEWithLogitsLoss()


In [13]:
len(testDS)
for a, b in testDS:
    print( a.shape, b)
    break

torch.Size([3, 50, 50]) 0


In [14]:
# TRAINDL   = DataLoader(trainDS, batch_size=100) ## 학습용 데이터로더

In [15]:
# for feature, target in TRAINDL:
#     print(test(feature).shape)
#     print( target.reshape(-1,1).shape)
#     break

In [16]:
HIST = testCF.cycling()

torch.Size([103, 3, 50, 50])
torch.Size([103])

EPOCH[0/100]----------------
- TRAIN_LOSS 0.92259  ACC 0.60000
- VALID_LOSS 0.68608  ACC 0.42718
11.25초
[0] - num_bad_epochs : 0 
torch.Size([103, 3, 50, 50])
torch.Size([103])

EPOCH[1/100]----------------
- TRAIN_LOSS 0.91889  ACC 0.60000
- VALID_LOSS 0.68617  ACC 0.42718
6.75초
[1] - num_bad_epochs : 1 
torch.Size([103, 3, 50, 50])
torch.Size([103])

EPOCH[2/100]----------------
- TRAIN_LOSS 0.91897  ACC 0.60000
- VALID_LOSS 0.68495  ACC 0.42718
11.21초
[2] - num_bad_epochs : 0 
torch.Size([103, 3, 50, 50])
torch.Size([103])

EPOCH[3/100]----------------
- TRAIN_LOSS 0.92055  ACC 0.63333
- VALID_LOSS 0.68482  ACC 0.42718
7.09초
[3] - num_bad_epochs : 0 
torch.Size([103, 3, 50, 50])
torch.Size([103])

EPOCH[4/100]----------------
- TRAIN_LOSS 0.90968  ACC 0.56667
- VALID_LOSS 0.68455  ACC 0.42718
16.67초
[4] - num_bad_epochs : 0 
torch.Size([103, 3, 50, 50])
torch.Size([103])

EPOCH[5/100]----------------
- TRAIN_LOSS 0.91313  ACC 0.60000
-

In [17]:
len(HIST['Train'])

2